# W16-D5 · Brief 验收对账 × G-07 活事件（+7 孤儿表）× 挡板检测延迟模拟

**今天的主问题**：四条验收线三绿一黄，为什么最重要的发现（canonical 477→484，+7 张孤儿表）不来自任何一条验收线？

本 notebook 是可执行版对账：
- §1 验收线复算 scorecard——每条线**运行时重放证据**（sha256 / git / 门禁脚本 / 回执指纹），不是转述
- §2 G-07 mini 集合门——用真实 canonical 双版本（HEAD vs origin/main）跑「正/负/泄漏」三检原型，判出今天的 7 张孤儿表
- §3 检测延迟蒙特卡洛——量化「周 digest / 日探针(只数commit) / 日表门」三种节奏对同一漂移事件的检测延迟分布与孤儿积压

与 md 的分工：md 是阅读材料，本文只在真实数据上执行判决与模拟。

In [ ]:
# ---- 环境：中文字体（TOOLS.md 标准方式）+ 路径 ----
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

import hashlib, json, re, subprocess, sys
from pathlib import Path
import numpy as np

BASE = Path("/root/learning-notebooks")
SEM = BASE / "semantic-model"
GOV = SEM / "governance"
LNK = Path("/root/lnkcre")
DOCS = Path("/root/docs")

def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"命令失败 {cmd}: {r.stderr[:300]}")
    return r.stdout

def sha16(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:16]

print("路径就绪")

## §1 验收线复算 scorecard（brief §3 逐条，今日重放）

验收线的正确形态 = **可重放断言**（D4 evidence-chain 的推广）：每条线带「复核动作 / 期望 / 实测 / 判决」。
下面四条线的证据全部在本 cell 运行时重算——包括 SoT 指纹、docs 增量主题扫描、两道锚点门实跑、pack 磁盘指纹 vs 回执对账。

In [ ]:
# ---- §1 四条验收线：运行时复算 ----
lines = []

# ① G-01：SoT 指纹（活算）+ docs 增量 ontology 主题计数（活算）+ 家族指纹（活算）
sot = DOCS / "lanlnk/config/ontology/business-ontology.yaml"
c1_actual = sha16(sot)
behind_docs = int(run("git rev-list --count HEAD..origin/main", cwd=DOCS).strip())
docs_topics = run("git log --oneline HEAD..origin/main", cwd=DOCS)
ontology_hits = len(re.findall(r"ontology|governance", docs_topics, re.I))
five = {
  "business-ontology": sha16(sot),
  "lnkchat-30products": sha16(DOCS/"lanlnk/30-products/lnkchat/ontology.yaml"),
  "LnkChatBI-out": sha16(DOCS/"lanlnk/out/prd/LnkChatBI/output/ontology.yaml"),
  "lnkreport-out": sha16(DOCS/"lanlnk/out/prd/lnkreport/output/ontology.yaml"),
  "legacy-frozen": sha16(next((DOCS/"lanlnk/90-legacy").rglob("ontology.yaml"))),
}
expected_five = {"business-ontology":"bf550bc24de66813","lnkchat-30products":"0a74edf9be475ecc",
  "LnkChatBI-out":"0240e95e2f9a84a1","lnkreport-out":"973f3b24a0e26d16","legacy-frozen":"e17ad8395baf8386"}
five_ok = all(five[k]==v for k,v in expected_five.items())
lines.append(("①", "立案受理+对账报告",
  f"对账侧：SoT {c1_actual}（基线 bf550bc24de66813，{'一致' if c1_actual=='bf550bc24de66813' else '漂移!'}）；"
  f"五件家族指纹 {'5/5 零漂移' if five_ok else '漂移!'}；docs behind {behind_docs} 中 ontology/governance 主题 {ontology_hits} 处",
  "部分达成（受理在主仓侧，对账侧 100%）"))

# ② frozen CI 红绿可测：两门 + selftest 实跑
g05 = run("python3 ci/frozen_effect_ci.py anchors --anchors g05-effect-anchors.yaml --repo /root/lnkcre", cwd=GOV)
g05_green = "GREEN" in g05 and g05.strip().splitlines()[-1]
ident = run("python3 ci/identity_anchor_ci.py verify --anchors identity-analysis-anchors.yaml --repo /root/lnkcre", cwd=GOV)
ident_green = "GREEN" in ident and ident.strip().splitlines()[-1]
st = run("python3 ci/identity_anchor_ci.py selftest --anchors identity-analysis-anchors.yaml --repo /root/lnkcre", cwd=GOV)
st_green = "GREEN" in st
lines.append(("②", "frozen CI 红绿可测+突变留痕",
  f"G-05 门：{g05.strip().splitlines()[-1]}；Identity 门：{ident.strip().splitlines()[-1]}；selftest：{'GREEN' if st_green else 'FAIL'}",
  "达成（超额：D4 升级含突变留痕）" if (g05_green and ident_green and st_green) else "复跑失败!"))

# ③ Identity 登记 + pack 回执对账（磁盘 pack 指纹 vs 回执登记——活算）
anch = yaml_safe = None
import yaml
anch = yaml.safe_load((GOV/"identity-analysis-anchors.yaml").read_text())
n_obj = anch["counts"]["total"]
receipt = json.loads((SEM/"consumers/lnkchatbi/import-pack/import-receipt.json").read_text())
fp_reg = receipt["pack"]["fingerprints"]
fp_disk = {"terminology.json": sha16(SEM/"consumers/lnkchatbi/import-pack/terminology.json"),
           "sql_examples.json": sha16(SEM/"consumers/lnkchatbi/import-pack/sql_examples.json")}
fp_ok = all(fp_disk[k]==fp_reg[k] for k in fp_reg)
lines.append(("③", "20/20 登记 + 值域11/11 + pack 重出回执",
  f"登记 {n_obj}/20 对象；pack v{receipt['pack']['version']}：术语{receipt['pack']['terms']}组/{receipt['pack']['aliases']}别名，"
  f"示例{receipt['pack']['sql_examples']}条；磁盘指纹 vs 回执 {'一致(8137b094…/0af3cb1f…)' if fp_ok else '不一致!'}；"
  f"幂等收敛 {receipt['upsert']['convergence']}；e2e {receipt['coverage_ab']['e2e_exec_ok']}/11（值域 11/11 见生成器自检，09-18 复现）",
  "达成" if (n_obj==20 and fp_ok) else "复跑失败!"))

# ④ digest 落盘 + 第二 ontology 裁决（文件存在 + 裁决行数 + 指纹活算）
dg = SEM/"sync/digest-2026-W38.md"
dg_ok = dg.exists() and ("产品级 ontology 家族治理裁决" in dg.read_text())
second_fp = sha16(DOCS/"lanlnk/30-products/lnkchat/ontology.yaml")
lines.append(("④", "digest 落盘 + 第二 ontology 裁决",
  f"digest-2026-W38.md {'存在且含 §5 裁决' if dg_ok else '缺失!'}；第二 ontology.yaml 今日指纹 {second_fp}"
  f"（审计基线 0a74edf9be475ecc，{'未漂移' if second_fp=='0a74edf9be475ecc' else '漂移!'}）",
  "达成" if (dg_ok and second_fp=="0a74edf9be475ecc") else "复跑失败!"))

print("=" * 100)
for no, name, actual, verdict in lines:
    print(f"[{no}] {name}\n    实测：{actual}\n    判决：{verdict}")
print("=" * 100)
n_green = sum(1 for *_, v in lines if v.startswith("达成"))
print(f"对账结论：{n_green}/4 线达成（①为部分达成——我方侧 100%，受理侧 0%）")
assert n_obj == 20 and fp_ok and g05_green and ident_green and st_green, "验收线复算失败"
print("§1 断言全过")

## §2 G-07 mini 集合门：今天的活事件判决（真实数据，三检原型）

语义层 v0.1 是索引层（分层 SoT 宪章：不复制下游源），其「表宇宙登记锚」= 最近一次周验基线的 canonical 快照（W39 基线 477）。
G-07 门的三检原型（集合版，对应 Identity 三件套）：
- **正锚**：canonical 新表 ∈ 登记宇宙 ∪ citing change → 否则 BROKEN
- **负锚**：登记宇宙的表从 canonical 消失 → BROKEN（登记腐烂）
- **泄漏检查**：新表落地却无任何语义层 change 引用 → BROKEN（今天事件的形态）

In [ ]:
# ---- §2 mini 集合门：HEAD vs origin/main 的 canonical 双版本 ----
canon_path = "backend/internal/platform/database/testdata/canonical_tables.txt"
head_tables = set(run(f"git show HEAD:{canon_path}", cwd=LNK).split())
origin_tables = set(run(f"git show origin/main:{canon_path}", cwd=LNK).split())
print(f"canonical：HEAD(基线/登记宇宙) {len(head_tables)} 表 | origin/main(现实) {len(origin_tables)} 表")

# 语义模型 v0.1 全文（索引层）——检查 +7 表是否有任何登记痕迹（含 Context anchors/术语层）
sm_text = (SEM/"mi-cre-semantic-model-v0.1.yaml").read_text()
model_table_total = 472  # yaml meta 自报（08-28 快照口径）

orphans = sorted(origin_tables - head_tables)     # 正锚判红对象
dead = sorted(head_tables - origin_tables)        # 负锚判红对象
changes_dir = SEM/"changes"
citing_texts = " ".join(p.read_text() for p in changes_dir.rglob("*.md")) if changes_dir.exists() else ""

print(f"\n== G-07 mini 门判决（fail-closed 集合版）==")
print(f"[正锚] 新表 {len(orphans)} 张：")
n_broken_pos = 0
for t in orphans:
    in_model = t in sm_text
    cited = t in citing_texts
    verdict = "OK(citing)" if cited else ("DRIFTED(模型有痕迹,无citing)" if in_model else "BROKEN(孤儿)")
    n_broken_pos += verdict.startswith("BROKEN")
    print(f"    [{verdict}] {t}")
print(f"[负锚] 消失表 {len(dead)} 张：{'BROKEN(登记腐烂)' if dead else 'OK（零消失）'}")
leak = [t for t in orphans if t not in citing_texts]
print(f"[泄漏] 无 citing change 引用的新表 {len(leak)}/{len(orphans)} → {'BROKEN' if leak else 'OK'}")

print(f"\n「三数一表」口径披露：模型 yaml 自报 table_total=472（08-28 快照）| W39 基线快照 477 | origin 现实 {len(origin_tables)}")
print("（计数口径未声明 = 883/963 教训的表级重演——G-07 草稿已把「口径声明」写进 frontmatter 提案）")

# 可视化：表宇宙漂移时间线 + 孤儿族
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
dates = ["08-28", "09-16", "09-18(未pull)"]
counts = [472, 477, len(origin_tables)]
bars = ax.bar(dates, counts, color=["#9aa5b1", "#5b8def", "#e4572e"], width=0.55)
for b, c in zip(bars, counts):
    ax.text(b.get_x()+b.get_width()/2, c+1, str(c), ha="center", fontsize=11)
ax.set_ylim(460, 492); ax.set_title("canonical 表宇宙漂移（快照口径 vs 现实）", fontsize=12)
ax.set_ylabel("表数")
ax2 = axes[1]
fam = {"leasing_policy 族(000227)": 5, "unit_pricing 族(000229)": 2}
ax2.barh(list(fam), list(fam.values()), color="#e4572e", height=0.45)
for i, (k, v) in enumerate(fam.items()):
    ax2.text(v+0.06, i, f"{v} 张孤儿", va="center", fontsize=10)
ax2.set_xlim(0, 7); ax2.set_title("+7 孤儿表按迁移族分布（PG migrations-pg）", fontsize=12)
plt.tight_layout(); plt.savefig(BASE/"第16周/w16d5_canonical_drift.png", dpi=130); plt.show()
print("图已存 第16周/w16d5_canonical_drift.png")
assert len(orphans) == 7 and not dead and len(leak) == 7, "活事件判决与 md 记载不符"
print("§2 断言全过：7 孤儿 / 0 死登记 / 7 泄漏——今天事件的 mini 门判决全红")

## §3 检测延迟蒙特卡洛：周 digest vs 日探针（只数 commit）vs 日表门

事件率用真实数据标定：08-28→09-18（21 天）canonical 总增量 +12 张 ≈ **λ=0.57 张/天** 的泊松到达。
三种防守节奏对「单张新表落地→被语义层看见」的延迟：
- **W 周频 digest**（现状 S1-canonical 项，W39 才首跑）：每 7 天一次全对
- **P 日探针只数 commit**（现状 S2）：看得见 commit 看不见表——表事件实际仍要等 digest（对表盲）
- **G 日表门**（G-07 升级）：每天 canonical 集合 diff，≤1 天必见
模拟 90 天 × 5000 次，看延迟分布与孤儿积压轨迹。

In [ ]:
# ---- §3 蒙特卡洛：检测延迟与孤儿积压 ----
rng = np.random.default_rng(42)
lam = 12 / 21                      # 张/天（真实标定）
DAYS, TRIALS = 90, 5000
STRATS = {"W 周digest": 7, "P 日探针(对表盲)": 7, "G 日表门": 1}  # P 对表事件等价 W——这正是要点

lat = {k: [] for k in STRATS}
backlog_p90 = {k: [] for k in STRATS}
for k, period in STRATS.items():
    all_backlogs = np.zeros((TRIALS, DAYS))
    for tr in range(TRIALS):
        arrivals = rng.poisson(lam, DAYS)
        backlog = np.zeros(DAYS)
        pending = []                      # 未检测表的到达日
        for d in range(DAYS):
            pending.extend([d] * arrivals[d])
            if (d + 1) % period == 0 or d == DAYS - 1:   # 周期检测点（期末强制一次，避免删失）
                for a in pending:
                    lat[k].append((d - a) + rng.uniform(0.5, 1.0))  # 到达日→检测日的真实间隔+日内粒度
                pending = []
            backlog[d] = len(pending)
        all_backlogs[tr] = backlog
    backlog_p90[k] = np.percentile(all_backlogs, 90, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
ax = axes[0]
colors = {"W 周digest": "#5b8def", "P 日探针(对表盲)": "#f0a202", "G 日表门": "#2ca02c"}
for k in STRATS:
    v = np.sort(np.array(lat[k]))
    cdf = np.arange(1, len(v) + 1) / len(v)
    ax.plot(v, cdf, label=f"{k}（均值 {v.mean():.1f} 天 / p95 {np.percentile(v,95):.0f} 天）", color=colors[k], lw=2)
ax.set_xlabel("新表落地 → 语义层看见 的延迟（天）"); ax.set_ylabel("CDF")
ax.set_title(f"检测延迟分布（λ={lam:.2f} 张/天, {DAYS}天×{TRIALS}次）", fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax = axes[1]
for k in STRATS:
    ax.plot(backlog_p90[k], label=k, color=colors[k], lw=2)
ax.set_xlabel("天数"); ax.set_ylabel("未检测孤儿表数（p90）")
ax.set_title("孤儿积压轨迹（p90）——周频节奏下的稳定负债带", fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(BASE/"第16周/w16d5_detection_latency.png", dpi=130); plt.show()
print("图已存 第16周/w16d5_detection_latency.png")

w = np.array(lat["W 周digest"]); g = np.array(lat["G 日表门"])
print(f"结论数字：周频均值 {w.mean():.1f} 天 vs 日表门 {g.mean():.1f} 天；")
print(f"最坏延迟：周频 {w.max():.0f} 天（=整周盲窗）vs 日表门 {g.max():.0f} 天；")
print(f"p90 孤儿积压峰值：周频 {backlog_p90['W 周digest'].max():.0f} 张 vs 日表门 {backlog_p90['G 日表门'].max():.0f} 张")
print("要点：P（只数 commit 的探针）对表事件的延迟曲线与 W 完全重合——速度是假象，看见的才作数。")
assert w.mean() > 3 and g.mean() <= 1.0 + 1e-9 and abs(w.mean() - np.array(lat['P 日探针(对表盲)']).mean()) < 0.5
print("§3 断言全过")

## §4 结论

1. **验收线复算 ≠ 任务回忆**：四条线全部运行时重放（指纹/门禁/回执对账），三绿一黄的「黄」不在我方手里——验收线的合格线设计必须区分「我方断言」与「他方事件」。
2. **活事件判决**：+7 孤儿 / 0 死登记 / 7 泄漏——G-07 mini 集合门在真实数据上第一次工作就抓到现行，正/负/泄漏三检结构成立。
3. **节奏即风险**：日表门把最坏 7 天盲窗压到 1 天，孤儿积压峰值从 ~5 张压到 ~1 张；「只数 commit」的日探针对表事件零增益——G-07 草稿 §3 的探针升级由此而来。
4. 明日（D6）：把 §2 原型转正为 `canonical_drift_ci.py`（selftest 注入排练 + 真实事件回放用例）。

In [ ]:
# ---- §4 汇总断言 ----
summary = {
  "验收线": {"①": "部分达成(对账100%/受理0%)", "②": "达成(超额)", "③": "达成", "④": "达成"},
  "G07活事件": {"孤儿表": len(orphans), "死登记": len(dead), "无citing泄漏": len(leak),
                "明细": orphans},
  "canonical": {"模型自报(08-28)": model_table_total, "W39基线(09-16)": len(head_tables),
                "origin现实(09-18)": len(origin_tables)},
  "蒙特卡洛": {"周频均值天": round(float(w.mean()), 1), "日表门均值天": round(float(g.mean()), 1),
               "周频最坏天": int(w.max()), "日表门最坏天": int(g.max())},
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert len(orphans) == 7 and n_green >= 3
print("\nW16-D5 notebook 全部断言通过 ✓")